<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/appendix_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch入門

## A.2テンソルを理解する
- テンソルはその次数（階数）で特徴づけることができる数学的なオブジェクト
- スカラー（単なる数値）は階数0のテンソル
- ベクトルは階数1のテンソル
- 行列は階数2のテンソル

In [ ]:
# PyTorchテンソルを作成する
import torch

tensor0d = torch.tensor(1) # 0次元テンソル（スカラー）を作成

tensor1d = torch.tensor([1, 2, 3]) # 1次元テンソル（ベクトル）を作成

tensor2d = torch.tensor([[1, 2],
                         [3, 4]]) # 2次元テンソルを作成

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]]) # 3次元テンソルを作成

In [ ]:
# テンソルのデータ型
print(tensor1d.dtype)

In [ ]:
# 浮動小数点からテンソルを作成
# 32ビットの精度のテンソルを作成

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

- 32ビット浮動小数点は、64ビット浮動小数点よりも少ないメモリリソースと計算リソースでほとんどのディープラーニングに十分な精度を実現
- GPUアーキテクチャは32ビット計算に合わせて最適化されているため、このデータ型を使用するとモデルの訓練と推論を大幅に高速化できる
- テンソルの`to()` メソッドを使って精度の変更もできる

In [ ]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

In [ ]:
# PyTorchの一般的なテンソル演算
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])
print(tensor2d)

In [ ]:
print(tensor2d.shape)

In [ ]:
# reshape
print(tensor2d.reshape(3, 2))

In [ ]:
# reshapeより view()メソッドの方が一般的
print(tensor2d.view(3, 2))

- `view()` の場合は元のデータが連動していなければならず、そうでない場合エラーになる
- `reshape()` の場合は元の形状に関係なく動作し、必要であればデータをコピーすることで、目的の形状を確保する

In [ ]:
# テンソルの転置
print(tensor2d)
print(tensor2d.T)

In [ ]:
# 行列の乗算（mamul()）
print(tensor2d.matmul(tensor2d.T))

In [ ]:
# @演算子での行列乗算
print(tensor2d @ tensor2d.T)

## A.3 モデルを計算グラフとしてとらえる
- 自動微分エンジンであるautograd
- 動的計算グラフを使用して勾配を自動で計算する
- 計算グラフ：数式の表現と可視化を可能にする有効グラフのこと
- 誤差逆伝播法に必要な勾配を計算するのに使用する

In [ ]:
# ロジスティック回帰のフォワードパス
import torch.nn.functional as F

y = torch.tensor([1.0]) # 正解ラベル
x1 = torch.tensor([1.1]) # 入力特徴量
w1 = torch.tensor([2.2]) # 重みパラメータ
b = torch.tensor([0.0]) # バイアスユニット
z = x1 * w1 + b # 総入力
a = torch.sigmoid(z) # 活性化と出力
loss = F.binary_cross_entropy(a, y)

## A.4 自動微分は行内の計算を簡易化する
- 終端（葉）ノードでrequired_grad属性がTrueに設定されていれば、デフォルトで内部に計算グラフが構築される
- 連鎖律で誤差逆伝播を考える
- 偏微分：ある関数がその変数の1つに対してどのように変化するかを表す尺度
- 勾配：多変量関数の偏微分をすべて含んでいるベクトル
- 多変量関数：入力として複数の変数を持つ関数

In [ ]:
# autogradによる勾配の計算
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

# 計算グラフを残すためにretain_graf=Trueを指定
# デフォルトはメモリ解放するために計算グラフを削除する
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [ ]:
# モデルパラメータ
print(grad_L_w1)
print(grad_L_b)

In [ ]:
# 自動化
loss.backward()
print(w1.grad)
print(b.grad)

## A.5 多層ニューラルネットワークを実装する
- 多層パーセプトロン：全結合ネットワーク

In [ ]:
# 2つの隠れ層をもつ多層パーセプトロン
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1つ目の隠れ層
            torch.nn.Linear(num_inputs, 30),
            # 非線形活性化関数は隠れ層の間に置かれる
            torch.nn.ReLU(),

            # 2つ目の隠れ層
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # 出力層
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)

        # 最後の層の出力はロジットと呼ばれる
        return logits

In [ ]:
# インスタンス化
model = NeuralNetwork(50, 3)

In [ ]:
print(model)

In [ ]:
# パラメータの総数を確認
num_paramas = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {num_paramas}")

In [ ]:
# インデックス位置0にある重みパラメータ
print(model.layers[0].weight)

In [ ]:
# 形状
print(model.layers[0].weight.shape)

In [ ]:
# フィードフォワードパスの挙動
torch.manual_seed(123)
x = torch.rand((1, 50))
out = model(x)
print(out)

In [ ]:
# torch.no_grad()で勾配の追跡を停止
# メモリリソースや計算リソースが大幅に削減される
with torch.no_grad():
    out = model(x)
print(out)

In [ ]:
# 予測値のクラス所属確率の計算
with torch.no_grad():
    out = torch.softmax(model(x), dim=1)
print(out)

## A6. 効率的なデータローダーをセットアップする

In [ ]:
# 単純なデータセットを作成する
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [ ]:
# カスタムDatasetクラスを定義する
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        # データレコードと対応するラベルを1つだけ取得
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        # データセットの全体的な長さ
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
print(len(train_ds))

In [ ]:
# データローダーをインスタンス化する
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [ ]:
# バッチの出力
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}: {x}, {y}")

- train_loaderは訓練データセットを反復処理して、各訓練サンプルに1回ずつアクセスする -> 「訓練エポック」という
- バッチサイズが割り切れない場合に、訓練エポックの最後のバッチとしてかなり小さいバッチを使うと、訓練中の収束の妨げになることがある
- この問題を回避するために、`drop_last=True`を設定する
- 各エポックの最後のバッチが削除される

In [ ]:
# 最後のバッチを削除する訓練ローダー
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [ ]:
# 最後のバッチが消えていること確認
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}: {x}, {y}")

## A.7 典型的な訓練ループ

In [ ]:
# PyTorchでのニューラルネットワークの訓練
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3
for epoch in range(num_epochs):

    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad() # 勾配を累積したくないため、前のイテレーションの勾配を0に設定
        loss.backward() # 指定されたモデルのパラメータの損失の勾配を計算
        optimizer.step() # オプティマイザが勾配を使ってモデルパラメータを更新

        ### ロギング
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
            f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
            f" | Train Loss: {loss:.2f}")

        model.eval()

In [ ]:
# 練習問題 A-3: モデルのパラメータ数
# requires_grad=True: 学習可能なパラメータ
sum(p.numel() for p in model.parameters() if p.requires_grad)

- SGDオプティマイザの場合、勾配に学習率をかけて、スケール化された負の勾配をパラメータに足す

In [ ]:
# 訓練後はモデルを使って予測
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

In [ ]:
# クラスの所属確率を求めるために、softmax()関数を使う
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

In [ ]:
# argmax()関数を使用してクラスラベルを取得する
# dim=1を指定すると、各行において最も大きい値のインデックス位置が返される
# dim=0を指定すると、各列において最も大きい値のインデックス位置が返される
predictions = torch.argmax(probas, dim=1)
print(predictions)

In [ ]:
# クラスラベルを得るだけであればsoftmax()関数は不要
# softmax()関数はlogitsの正規化を行っている
predictions = torch.argmax(outputs, dim=1)
print(predictions)

In [ ]:
# モデルの予測が100%正しいこと確認
predictions == y_train

In [ ]:
# 正しい予測の数をカウントできる
torch.sum(predictions == y_train).item()

In [ ]:
# 予測正解率を計算する関数
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions # 正解ラベルと一致するかをTrue/Falseで返す
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item() # item()はテンソルの値をPythonのfloatで返す

In [ ]:
print(compute_accuracy(model, train_loader))

In [ ]:
print(compute_accuracy(model, test_loader))

## A.8 モデルの保存と読み込み

In [ ]:
torch.save(model.state_dict(), "model.pth")

- モデルのstate_dictはPythonのディクショナリオブジェクトであり、モデル内の各層とその訓練可能なパラメータ（重みとバイアス）をマッピングする

In [ ]:
# モデルのロード
model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("model.pth"))

## A.9 GPUを使って訓練性能を最適化する

In [ ]:
# GPUが使えるか
print(torch.cuda.is_available())

In [ ]:
# テンソルの演算
# 通常はCPU上で計算される
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

In [ ]:
# GPUで計算
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")
print(tensor_1 + tensor_2)

In [ ]:
# 一方がCPU、もう一方がGPUの場合は計算できない
tensor_1 = tensor_1.to("cpu")
print(tensor_1 + tensor_2)

In [ ]:
# GPU上での訓練ループ
torch.manual_seed(123)

model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda")
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ロギング
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
            f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
            f" | Train/Validation loss: {loss:.2f}")

    model.eval()

In [ ]:
torch.cuda.device_count()